# 📑 PageIndex — Vectorless RAG Crash Course
### Reasoning-based RAG with No Vector DB, No Chunking
**By Anjaneya**

---

## 🧠 What You'll Learn

| # | Topic |
|---|-------|
| 1 | Why Vector RAG fails on professional documents |
| 2 | How PageIndex builds a tree index from a PDF |
| 3 | LLM Tree Search — reasoning over structure |
| 4 | Full end-to-end Vectorless RAG pipeline |
| 5 | Expert-guided retrieval (domain knowledge injection) |
| 6 | Chat API — zero LLM setup |
| 7 | Self-hosted open-source option |
| 8 | Vector RAG vs PageIndex — Side-by-Side |

---

## 🔑 Key Concept

> **Traditional RAG** → chunk → embed → cosine similarity → retrieve  
> **PageIndex RAG** → build tree → LLM reasons over tree → retrieve exact sections

**The problem with vector RAG:**  
`Similarity ≠ Relevance`  
A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.

---

## 🏗️ Architecture Overview

```
┌─────────────────────────────────────────────────────┐
│                  VECTORLESS RAG PIPELINE             │
├─────────────┬───────────────────┬───────────────────┤
│  INDEXING   │    RETRIEVAL      │    GENERATION     │
│             │                   │                   │
│ PDF Upload  │  Query + Tree     │  Context Nodes    │
│     ↓       │       ↓           │       ↓           │
│ LLM reads   │  LLM reasons      │  LLM writes       │
│ structure   │  over tree        │  cited answer     │
│     ↓       │       ↓           │                   │
│ Tree Index  │  node_ids list    │                   │
│  (JSON)     │                   │                   │
└─────────────┴───────────────────┴───────────────────┘
```


---
## 📦 Section 1: Install & Setup

**What we do here:**
- Install PageIndex SDK + OpenAI
- Load API keys securely from `.env` (never hardcode keys!)
- Initialize both clients with a reusable config dataclass

> 🔑 Get your **PageIndex API key** from: https://dash.pageindex.ai/api-keys  
> 🔑 Get your **OpenAI API key** from: https://platform.openai.com

> ⚠️ **Security Note:** Never hardcode API keys in notebooks. Always use environment variables or `.env` files.


In [ ]:
# Install required packages
!pip install -U pageindex openai python-dotenv

In [ ]:
# ── Create a .env file (run this once) ──────────────────────────────────────
# Uncomment, fill in your keys, then run this cell ONCE

# env_content = """
# PAGEINDEX_API_KEY=your_pageindex_key_here
# OPENAI_API_KEY=your_openai_key_here
# """
# with open(".env", "w") as f:
#     f.write(env_content.strip())
# print("✅ .env file created")

In [ ]:
import os
import json
import time
import functools
from dataclasses import dataclass, field
from typing import Optional
from dotenv import load_dotenv

load_dotenv()

# ── Load all keys from .env — never hardcode API keys ───────────────────────
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing — add PAGEINDEX_API_KEY to .env")
print("OpenAI key loaded:   ", "✅" if OPENAI_API_KEY    else "❌ Missing — add OPENAI_API_KEY to .env")

In [ ]:
# ── Centralised RAG Configuration ───────────────────────────────────────────
# One place to control every tunable knob in the pipeline.
# Change values here and every function below picks them up automatically.

@dataclass
class RAGConfig:
    """Central configuration for the Vectorless RAG pipeline."""
    # Models
    search_model:    str = "gpt-4o"        # model used for tree search
    generate_model:  str = "gpt-4o"        # model used for answer generation

    # Tree compression
    summary_chars:   int = 150             # chars from each node summary kept during compression

    # Retrieval
    max_nodes:       int = 10              # cap on nodes returned per query

    # Polling
    poll_interval_s: int = 5              # seconds between status checks
    max_poll_tries:  int = 60             # give up after 5 minutes (60 × 5s)

    # Retry
    max_retries:     int = 3              # API call retries on transient errors
    retry_delay_s:   float = 2.0          # seconds to wait between retries


# Singleton config used across the notebook — override any field as needed
cfg = RAGConfig()
print("✅ RAGConfig ready:", cfg)

In [ ]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ PageIndex client ready")
print("✅ OpenAI client ready")

---
## 🌲 Section 2: Upload & Index a PDF

**What happens here:**
1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical **tree index** (like a smart Table of Contents)
4. Returns a `doc_id` for all future operations

**Why NO chunking?**  
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.


In [ ]:
# ── Upload your PDF ──────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./sample_document.pdf"   # ← change this

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"PDF not found at '{PDF_PATH}'. Update PDF_PATH above.")

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

In [ ]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# Improved: bounded polling with timeout so the notebook never hangs forever.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

for attempt in range(cfg.max_poll_tries):
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   [{attempt + 1}/{cfg.max_poll_tries}] Status: {status}")

    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        raise RuntimeError("❌ Processing failed. Check your PDF format or PageIndex dashboard.")

    time.sleep(cfg.poll_interval_s)
else:
    raise TimeoutError(f"❌ Indexing did not complete within {cfg.max_poll_tries * cfg.poll_interval_s}s.")

---
## 🔍 Section 3: Inspect the Tree Structure

**What the tree looks like:**

```
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**


In [ ]:
# ── Fetch the full tree ──────────────────────────────────────────────────────
tree_result    = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

In [ ]:
# ── Pretty-print the full tree ───────────────────────────────────────────────

def print_tree(nodes: list, indent: int = 0) -> None:
    """Recursively pretty-print tree node titles and page numbers."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)


print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

In [ ]:
# ── Count total nodes ────────────────────────────────────────────────────────

def count_nodes(nodes: list) -> int:
    """Recursively count all nodes in the tree."""
    return sum(1 + (count_nodes(n["nodes"]) if n.get("nodes") else 0) for n in nodes)


total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

---
## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

### Vector RAG retrieval:
```
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```
*Problem: finds what's similar, not what's relevant*

### PageIndex retrieval:
```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**

### Improvements in this version:
- ✅ `compress_tree()` extracted as a **shared utility** (not duplicated in every function)
- ✅ Retry logic with exponential back-off on transient API errors
- ✅ Result capped by `cfg.max_nodes` to avoid over-retrieval


In [ ]:
# ── Shared utilities ─────────────────────────────────────────────────────────
# Extracted once so every search function reuses the same logic.

def compress_tree(nodes: list, summary_chars: int = None) -> list:
    """
    Compress the full tree to a lightweight representation for the LLM prompt.
    Keeps only node_id, title, page, and a short summary snippet.

    Args:
        nodes:        Raw tree from PageIndex.
        summary_chars: Max characters from node text to include (default: cfg.summary_chars).

    Returns:
        Compressed list of dicts safe to pass in a prompt.
    """
    chars = summary_chars or cfg.summary_chars
    out = []
    for n in nodes:
        # Use 'summary' field if available, fall back to 'text'
        snippet = (n.get("summary") or n.get("text") or "")[:chars]
        entry = {
            "node_id": n["node_id"],
            "title":   n["title"],
            "page":    n.get("page_index", "?"),
            "summary": snippet,
        }
        if n.get("nodes"):
            entry["children"] = compress_tree(n["nodes"], chars)
        out.append(entry)
    return out


def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect all nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found


def with_retry(fn, *args, retries: int = None, delay: float = None, **kwargs):
    """
    Call fn(*args, **kwargs) with simple retry logic.
    Retries on any exception up to cfg.max_retries times.
    """
    retries = retries if retries is not None else cfg.max_retries
    delay   = delay   if delay   is not None else cfg.retry_delay_s
    for attempt in range(retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt == retries:
                raise
            wait = delay * (2 ** attempt)  # exponential back-off
            print(f"   ⚠️ Attempt {attempt+1} failed ({e}). Retrying in {wait:.1f}s...")
            time.sleep(wait)


print("✅ Shared utilities defined: compress_tree, find_nodes_by_ids, with_retry")

In [ ]:
# ── LLM Tree Search ──────────────────────────────────────────────────────────

def llm_tree_search(
    query: str,
    tree: list,
    model: str = None,
    expert_rules: Optional[str] = None,
) -> dict:
    """
    Core PageIndex retrieval.
    Sends the compressed tree + query to an LLM and asks it to reason
    about which node_ids most likely contain the answer.

    Args:
        query:        User question.
        tree:         Full PageIndex tree.
        model:        OpenAI model (default: cfg.search_model).
        expert_rules: Optional domain-expert routing rules to inject.

    Returns:
        Dict with 'thinking' (LLM reasoning) and 'node_list' (node IDs).
    """
    model = model or cfg.search_model
    compressed = compress_tree(tree)

    expert_block = ""
    if expert_rules:
        expert_block = f"\n\nExpert Routing Rules (follow these carefully):\n{expert_rules}"

    prompt = f"""You are a domain expert analyzing a document tree structure.
Identify the node IDs that most likely contain the answer to the query below.
Think step-by-step about which sections are relevant.
Return at most {cfg.max_nodes} node IDs.

Query: {query}

Document Tree:
{json.dumps(compressed, indent=2)}{expert_block}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    def _call():
        response = openai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        result = json.loads(response.choices[0].message.content)
        # Cap node list
        result["node_list"] = result.get("node_list", [])[:cfg.max_nodes]
        return result

    return with_retry(_call)


print("✅ llm_tree_search() defined — supports optional expert_rules parameter")

In [ ]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is the syllabus covered in Modern LLM finetuning?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

---
## ⚙️ Section 5: Full End-to-End RAG Pipeline

**3 steps:**
1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes  
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**
- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly *which section* the answer comes from
- No hallucination from irrelevant chunks

**Improvements in this version:**
- ✅ Token-usage tracking per call
- ✅ Result object returned (not just a string) — includes nodes, reasoning, and answer


In [ ]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(
    query: str,
    nodes: list,
    model: str = None,
) -> tuple[str, int]:
    """
    Takes retrieved nodes as context and generates a grounded, cited answer.

    Returns:
        (answer_text, total_tokens_used)
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document.", 0

    model = model or cfg.generate_model

    context_parts = [
        f"[Section: '{n['title']}' | Page {n.get('page_index', '?')}]\n"
        f"{n.get('text') or n.get('summary') or 'Content not available.'}"
        for n in nodes
    ]
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses like (Section: 'Title' | Page N).
Be concise and precise. If the context doesn't contain sufficient information, say so.

Question: {query}

Context:
{context}

Answer:"""

    def _call():
        response = openai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
        )
        answer = response.choices[0].message.content
        tokens = response.usage.total_tokens if response.usage else 0
        return answer, tokens

    return with_retry(_call)


print("✅ generate_answer() defined — returns (answer, token_count)")

In [ ]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(
    query: str,
    tree: list,
    expert_rules: Optional[str] = None,
    verbose: bool = True,
) -> dict:
    """
    Full end-to-end PageIndex RAG pipeline.

    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer

    Args:
        query:        User question.
        tree:         PageIndex tree.
        expert_rules: Optional routing rules for expert-guided retrieval.
        verbose:      Print step-by-step progress.

    Returns:
        Dict with keys: answer, node_ids, nodes, reasoning, tokens_used
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    # Step 1: Tree Search
    search_result = llm_tree_search(query, tree, expert_rules=expert_rules)
    node_ids      = search_result.get("node_list", [])
    reasoning     = search_result.get("thinking", "")

    if verbose:
        print(f"\n🧠 Reasoning: {reasoning[:250]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")

    # Step 3: Generate answer
    answer, tokens = generate_answer(query, nodes)

    if verbose:
        print(f"\n📝 Answer:\n{answer}")
        print(f"\n🔢 Tokens used: {tokens}")

    return {
        "answer":     answer,
        "node_ids":   node_ids,
        "nodes":      nodes,
        "reasoning":  reasoning,
        "tokens_used": tokens,
    }


print("✅ vectorless_rag() defined — returns full result dict")

In [ ]:
# ── Run the full pipeline ────────────────────────────────────────────────────
result = vectorless_rag(
    query="What are the topics covered in modern LLM finetuning?",
    tree=pageindex_tree,
)

In [ ]:
# ── Batch query runner with token tracking ───────────────────────────────────
# Improved: collects total tokens across all queries so you can monitor API cost.

test_queries = [
    "What are the topics covered in modern LLM finetuning?",
    "What are the topics covered in RAG?",
    "Summarize the Tokenization Deep Dive syllabus.",
]

total_tokens = 0

for q in test_queries:
    print()
    res = vectorless_rag(q, pageindex_tree, verbose=False)
    total_tokens += res["tokens_used"]
    print(f"Q: {q}")
    print(f"A: {res['answer'][:300]}...")
    print(f"   Tokens: {res['tokens_used']}")
    print("-" * 55)

print(f"\n📊 Total tokens across all queries: {total_tokens}")

---
## 🎓 Section 6: Expert-Guided Retrieval

**The killer feature no one talks about.**

With vector RAG, injecting domain expertise requires **fine-tuning the embedding model** — expensive and time-consuming.

With PageIndex, you just **add rules to the prompt**:

```
"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks  → check Part I, Item 1A"
```

This makes PageIndex instantly adaptable to any domain without any model training.

**Improvement:** Expert-guided retrieval is now built into `llm_tree_search()` and `vectorless_rag()` via the `expert_rules` parameter — no need for a separate function.


In [ ]:
# ── Expert Routing Rules — Advanced Route of Learning AI ─────────────────────

AI_COURSE_EXPERT_RULES = """
Route queries to the correct module using these rules:

M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training

Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""

print("✅ Expert rules defined")
print("   These are injected into the retrieval prompt at query time.")

In [ ]:
# ── Compare: with vs without expert rules ────────────────────────────────────
query = "Details of the modern LLM finetuning?"

print(f"🔍 Query: {query}\n")

print("── Without Expert Rules ──")
basic  = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print()

print("── With Expert Rules ──")
guided = llm_tree_search(query, pageindex_tree, expert_rules=AI_COURSE_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

In [ ]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────
# Expert rules now passed directly into vectorless_rag() — no wrapper needed.

result = vectorless_rag(
    query="Details of the syllabus of modern LLM finetuning",
    tree=pageindex_tree,
    expert_rules=AI_COURSE_EXPERT_RULES,
)
print(result["answer"])

---
## 💬 Section 7: Chat API — Zero LLM Setup

**When to use this:**
- You don't want to manage OpenAI API calls yourself
- You want a quick Q&A interface over your document
- You're building a chat product and want PageIndex to handle everything

PageIndex provides its own LLM — you just pass a question and `doc_id`.


In [ ]:
# ── Single question with Chat API ────────────────────────────────────────────
# No OpenAI key needed — PageIndex runs the LLM internally

question = "What are the key findings in this document?"

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id,
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

In [ ]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Improved: history stored inside the function as a list, not a global variable.

class DocumentChat:
    """Stateful multi-turn chat session with a PageIndex document."""

    def __init__(self, doc_id: str):
        self.doc_id  = doc_id
        self.history: list[dict] = []

    def ask(self, user_message: str) -> str:
        """Send a message and get a reply, maintaining full conversation history."""
        self.history.append({"role": "user", "content": user_message})

        response = pi_client.chat_completions(
            messages=self.history,
            doc_id=self.doc_id,
        )

        reply = response["choices"][0]["message"]["content"]
        self.history.append({"role": "assistant", "content": reply})
        return reply

    def reset(self) -> None:
        """Clear conversation history to start a fresh session."""
        self.history = []
        print("🔄 Conversation history cleared.")


# Simulate a 3-turn conversation
chat = DocumentChat(doc_id)

questions = [
    "What modules are covered in this course?",
    "Which of those is most relevant for someone interested in fine-tuning?",
    "What prerequisites would I need for those modules?",
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat.ask(q)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)

---
## 🛠️ Section 8: Self-Hosted Open Source Option

**Use this when:**
- You don't want to send documents to any cloud
- You need full data privacy / on-prem deployment
- You want to inspect or customize the tree-building logic

The open-source repo at https://github.com/VectifyAI/PageIndex lets you run the entire pipeline locally using your own OpenAI key.

**What the CLI does:**
1. Reads your PDF
2. Detects existing Table of Contents (if any)
3. Uses GPT-4o to build the hierarchical tree
4. Saves a `document_name_pageindex.json` alongside your PDF


In [ ]:
# ── Clone the open-source repo ───────────────────────────────────────────────
!git clone https://github.com/VectifyAI/PageIndex.git
%cd PageIndex
!pip install -r requirements.txt

In [ ]:
# ── Create .env for self-hosted mode ─────────────────────────────────────────
# The local runner uses CHATGPT_API_KEY (not OPENAI_API_KEY)

openai_key = os.getenv("OPENAI_API_KEY", "")
if not openai_key:
    raise EnvironmentError("OPENAI_API_KEY not set in environment.")

with open(".env", "w") as f:
    f.write(f"CHATGPT_API_KEY={openai_key}\n")

print("✅ .env created for self-hosted mode")

In [ ]:
# ── Run PageIndex locally on a PDF ───────────────────────────────────────────
# Optional parameters you can customize:
#   --model                  OpenAI model (default: gpt-4o-2024-11-20)
#   --toc-check-pages        Pages to scan for existing TOC (default: 20)
#   --max-pages-per-node     Max pages per tree node (default: 10)
#   --if-add-node-summary    Include summaries in output (yes/no)

PDF_PATH = "/path/to/your/document.pdf"   # ← change this

!python run_pageindex.py \
    --pdf_path {PDF_PATH} \
    --model gpt-4o-2024-11-20 \
    --toc-check-pages 20 \
    --max-pages-per-node 10 \
    --if-add-node-summary yes

In [ ]:
# ── Load locally generated tree ──────────────────────────────────────────────
# Output is saved as: <your_pdf_name>_pageindex.json

TREE_JSON_PATH = "/path/to/your/document_pageindex.json"  # ← change this

with open(TREE_JSON_PATH, "r") as f:
    local_tree = json.load(f)

print(f"🌲 Local tree loaded: {count_nodes(local_tree)} total nodes")
print_tree(local_tree)

In [ ]:
# ── Run the same RAG pipeline on the local tree ──────────────────────────────
# Everything from Sections 4–6 works identically with local trees

query  = "Summarize the executive summary section."
result = vectorless_rag(query, local_tree)

---
## 📊 Section 9: Vector RAG vs PageIndex — Side-by-Side

### Architecture Comparison

| Aspect | Traditional Vector RAG | PageIndex (Vectorless RAG) |
|--------|------------------------|--------------------------|
| **Document prep** | Chunk into fixed pieces | Build hierarchical tree |
| **Indexing** | Embed each chunk | LLM reads structure |
| **Storage** | Vector database | JSON file |
| **Query processing** | Embed query → ANN search | LLM reasons over tree |
| **What's retrieved** | Flat anonymous chunks | Named sections + page refs |
| **Explainability** | ❌ Opaque similarity score | ✅ Traceable reasoning |
| **Domain expertise** | ❌ Needs embedding fine-tune | ✅ Add rules to prompt |
| **Infrastructure** | Pinecone / FAISS / ChromaDB | No vector DB needed |
| **Best for** | Short, diverse documents | Long, structured documents |
| **FinanceBench accuracy** | ~80% | **98.7%** |

### When to use which

**Use Vector RAG when:**
- Documents are short and varied (FAQs, product descriptions)
- Semantic paraphrase matching is important  
- You need sub-second retrieval on millions of documents

**Use PageIndex when:**
- Documents are long and professionally structured (reports, manuals, legal docs)
- You need traceable, cited answers
- Domain expertise should guide retrieval
- You want to avoid vector DB infrastructure


In [ ]:
# ── Quick comparison demo ────────────────────────────────────────────────────

print("=" * 55)
print("VECTOR RAG approach (conceptual):")
print("=" * 55)
print("""
query_vec = embed_model.encode("What are EBITDA risks?")
chunks    = vector_db.similarity_search(query_vec, k=5)

# Returns: 5 text fragments ranked by cosine distance
# Problem: may return "market risk" chunks, not EBITDA section
# No page numbers, no section context, opaque ranking
""")

print("=" * 55)
print("PAGEINDEX approach (actual):")
print("=" * 55)
print("""
result = llm_tree_search("What are EBITDA risks?", tree)

# Returns: node IDs like ["0007", "0012"]
# LLM reasoning: "EBITDA is discussed in MD&A section (node 0007)
#                 and footnotes in Financial Statements (node 0012)"
# Full traceability — section title + page number
# Domain expertise injected via expert_rules= parameter
""")

---
## 🧹 Section 10: Cleanup

Delete documents from the PageIndex cloud when you're done  
to keep your storage clean.


In [ ]:
# ── Delete document from cloud ───────────────────────────────────────────────
# WARNING: This permanently deletes the tree index.
# Comment this out if you want to reuse the doc_id later.

# pi_client.delete_document(doc_id)
# print(f"🗑️ Deleted document: {doc_id}")
print("ℹ️ Deletion commented out — uncomment when you're done with this doc_id")

---
## ✅ Summary

You've now built a complete **Vectorless RAG** system with PageIndex.

### What you built:

| Function | Purpose |
|---|---|
| `compress_tree()` | Shared utility — compress full tree for LLM prompts |
| `find_nodes_by_ids()` | Retrieve section content from tree by node ID |
| `with_retry()` | Resilient API calls with exponential back-off |
| `llm_tree_search()` | LLM reasons over tree — supports optional expert rules |
| `generate_answer()` | LLM produces cited, grounded answers + token count |
| `vectorless_rag()` | Full pipeline — returns structured result dict |
| `DocumentChat` | Stateful multi-turn chat session over a document |
| `RAGConfig` | Central config dataclass for all tunable parameters |

### Key improvements over the original:

- 🔐 **No hardcoded API keys** — all secrets loaded from `.env`
- 🔁 **Retry logic** with exponential back-off on transient API errors
- 🧱 **`RAGConfig` dataclass** — one place to tune all parameters
- 🔧 **`compress_tree()` extracted** — eliminates duplicate code
- 📊 **Token usage tracking** — monitor API cost per query and in bulk
- ⏱️ **Bounded polling** — indexing loop times out cleanly
- 🗂️ **`DocumentChat` class** — replaces global conversation history
- 📦 **`vectorless_rag()` returns a dict** — easy to inspect nodes, reasoning, tokens
- ✅ **Type hints** throughout for IDE support

### Key takeaways:

- `Similarity ≠ Relevance` — the fundamental flaw of vector search
- Tree-based reasoning gives you **traceable**, **accurate**, **explainable** retrieval
- Domain expertise injection is just **prompt engineering** — no model training needed
- 98.7% on FinanceBench vs ~80% for vector RAG

---

### 🔗 Resources
- GitHub: https://github.com/VectifyAI/PageIndex
- Docs: https://docs.pageindex.ai
- Chat Platform: https://chat.pageindex.ai
- Blog: https://pageindex.ai/blog/pageindex-intro

---
*Crash course by **Anjaneya***
